In [1]:
import pandas as pd
df = pd.read_csv("/kaggle/input/trip-advisor-hotel-reviews/tripadvisor_hotel_reviews.csv")
df = df.Review
df = df[:50]
df

0     nice hotel expensive parking got good deal sta...
1     ok nothing special charge diamond member hilto...
2     nice rooms not 4* experience hotel monaco seat...
3     unique, great stay, wonderful time hotel monac...
4     great stay great stay, went seahawk game aweso...
5     love monaco staff husband stayed hotel crazy w...
6     cozy stay rainy city, husband spent 7 nights m...
7     excellent staff, housekeeping quality hotel ch...
8     hotel stayed hotel monaco cruise, rooms genero...
9     excellent stayed hotel monaco past w/e delight...
10    poor value stayed monaco seattle july, nice ho...
11    nice value seattle stayed 4 nights late 2007. ...
12    nice hotel good location hotel kimpton design ...
13    nice hotel not nice staff hotel lovely staff q...
14    great hotel night quick business trip, loved l...
15    horrible customer service hotel stay february ...
16    disappointed say anticipating stay hotel monac...
17    fantastic stay monaco seattle hotel monaco

In [2]:
df.size

50

In [3]:
# using pretrained model
import gensim
model = gensim.models.KeyedVectors.load_word2vec_format('/kaggle/input/google-word2vec/GoogleNews-vectors-negative300.bin', binary=True)

In [4]:
import spacy
nlp = spacy.load('en_core_web_sm')
noun_adj_pairs =[]

for review in df:
    doc = nlp(review)

    noun_adj_pair = []
    for chunk in doc.noun_chunks:
        adj = []
        noun = ""
        for tok in chunk:
            if tok.pos_ == "NOUN":
                noun = tok.text
            if tok.pos_ == "ADJ":
                adj.append(tok.text)
        if noun and adj:
            noun_adj_pair.append([noun,adj])

    # expected output
    noun_adj_pairs.append(noun_adj_pair)
    
noun_adj_pairs[:2]

[[['parking', ['nice', 'expensive']],
  ['deal', ['good']],
  ['reviews', ['previous']],
  ['room', ['easy', 'little', 'disappointed', 'non', '-', 'existent']],
  ['size', ['nice']],
  ['pillows', ['stiff', 'high']],
  ['neighbors', ['noisy']],
  ['touch', ['nice']],
  ['shopping', ['great']],
  ['experience', ['overall', 'nice']]],
 [['member', ['special']],
  ['seattle', ['20th']],
  ['description', ['extra']],
  ['room', ['standard']],
  ['breakfast', ['mixed', 'free']],
  ['advertising', ['false']],
  ['description', ['advertised']],
  ['duty', ['hard']],
  ['spots', ['best']],
  ['hotel', ['convenient']],
  ['half', ['20th']],
  ['email', ['nice']],
  ['room', ['great']],
  ['building', ['high']],
  ['property', ['cleaner']],
  ['room', ['impressed', 'left']],
  ['trips', ['short']],
  ['screen', ['good', 'ac']],
  ['rates', ['clean', 'super', 'high']]]]

In [5]:
noun_adj_pairs[3]

[['hotel', ['wonderful']],
 ['area', ['excellent', 'short', 'main']],
 ['room', ['pet', 'friendly']],
 ['curtains', ['big', 'striped']],
 ['touch', ['nice']],
 ['lobby', ['free']],
 ['feature', ['great']],
 ['hotel', ['great', 'friendly', 'free', 'wireless']],
 ['palatte', ['lovely', 'eclectic']]]

In [6]:
topics = []

for item in noun_adj_pairs:
    for i in item:
        topics.append(i)

print(topics[:5])
len(topics)

[['parking', ['nice', 'expensive']], ['deal', ['good']], ['reviews', ['previous']], ['room', ['easy', 'little', 'disappointed', 'non', '-', 'existent']], ['size', ['nice']]]


359

In [7]:
# Define a list of words
words = []
for word in topics :
    if word[0] in model:
        words.append(word)
words[:5]

[['parking', ['nice', 'expensive']],
 ['deal', ['good']],
 ['reviews', ['previous']],
 ['room', ['easy', 'little', 'disappointed', 'non', '-', 'existent']],
 ['size', ['nice']]]

In [8]:
# Create an empty list to store the groups
groups = []

# Create a set to keep track of the pairs of words that have already been compared
compared_pairs = set()
checked_words = []

# Iterate over the words
for word1 in words:
    if word1[0] not in checked_words:
        # Create a new group for the current word
        group = [word1]
        checked_words.append(word1[0])

        # Iterate over the remaining words
        for word2 in words:
            # Skip the current word
            if word1[0] == word2[0] or word2[0] in checked_words:
                continue

            # Create a tuple containing the pair of words
            pair = tuple(sorted([word1[0], word2[0]]))

            # If the pair has already been compared, skip it
            if pair in compared_pairs:
                continue

            # Find the similarity score between the two words
            similarity = model.similarity(word1[0], word2[0])

            # If the similarity score is greater than 0.8, add the second word to the group
            if similarity > 0.5:
                group.append(word2)
                checked_words.append(word2[0])

            # Add the pair to the set of compared pairs
            compared_pairs.add(pair)

        # Add the group to the list of groups
        groups.append(group)


# Print the groups
for group in groups:
    print(group)

[['parking', ['nice', 'expensive']]]
[['deal', ['good']], ['deals', ['online']]]
[['reviews', ['previous']]]
[['room', ['easy', 'little', 'disappointed', 'non', '-', 'existent']], ['rooms', ['nice']], ['bathroom', ['busy']], ['lounge', ['wonderful', 'friendly', 'clean']]]
[['size', ['nice']]]
[['pillows', ['stiff', 'high']], ['mats', ['warwick']], ['carpets', ['tired', 'old', 'dirty']]]
[['neighbors', ['noisy']]]
[['touch', ['nice']], ['touches', ['little']]]
[['shopping', ['great']]]
[['experience', ['overall', 'nice']]]
[['member', ['special']]]
[['seattle', ['20th']]]
[['description', ['extra']]]
[['breakfast', ['mixed', 'free']], ['dinner', ['dressed']], ['lunch', ['good', 'eaten']]]
[['advertising', ['false']]]
[['duty', ['hard']]]
[['spots', ['best']]]
[['hotel', ['convenient']], ['restaurant', ['adequate']], ['hotels', ['better']]]
[['half', ['20th']]]
[['email', ['nice']]]
[['building', ['high']], ['buildings', ['lower', 'drab', 'panaroma']]]
[['property', ['cleaner']]]
[['trip

In [9]:
groups[0][0]

['parking', ['nice', 'expensive']]

In [10]:
sorted_groups = sorted(groups, key=len, reverse=True)
first_5_largest_groups = sorted_groups[:10]
print(first_5_largest_groups)

[[['evening', ['available']], ['night', ['steep']], ['weekend', ['late', 'free']], ['nights', ['awake']], ['morning', ['early']]], [['room', ['easy', 'little', 'disappointed', 'non', '-', 'existent']], ['rooms', ['nice']], ['bathroom', ['busy']], ['lounge', ['wonderful', 'friendly', 'clean']]], [['bed', ['large', 'comfortable']], ['couch', ['double']], ['sleep', ['good']], ['beds', ['double']]], [['pillows', ['stiff', 'high']], ['mats', ['warwick']], ['carpets', ['tired', 'old', 'dirty']]], [['breakfast', ['mixed', 'free']], ['dinner', ['dressed']], ['lunch', ['good', 'eaten']]], [['hotel', ['convenient']], ['restaurant', ['adequate']], ['hotels', ['better']]], [['thing', ['funny']], ['things', ['positive']], ['guess', ['expensive']]], [['city', ['large']], ['town', ['good', 'great', 'pricey']], ['downtown', ['17th', 'excellent']]], [['shops', ['great']], ['restaurants', ['uphill']], ['shop', ['best']]], [['husband', ['memorial', 'best']], ['friend', ['february', '4th']], ['spouse', ['

In [11]:
for item in first_5_largest_groups:
    for i in item:
        print(i[0])
    print('\n')

evening
night
weekend
nights
morning


room
rooms
bathroom
lounge


bed
couch
sleep
beds


pillows
mats
carpets


breakfast
dinner
lunch


hotel
restaurant
hotels


thing
things
guess


city
town
downtown


shops
restaurants
shop


husband
friend
spouse




In [12]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import numpy as np

# Download the required resources
#nltk.download('vader_lexicon')

# Initialize the sentiment analyzer
sia = SentimentIntensityAnalyzer()


for group in first_5_largest_groups:
    common_words = []
    adjectives = []
    for item in group:
        common_words.append(item[0])
        adjectives.extend(item[1])
    
    print(common_words)
    
    # Join the words into a sentence
    sentence = ' '.join(adjectives)

    # Compute the sentiment score of the sentence
    sentiment = sia.polarity_scores(sentence)['compound']

    # Compute the centroid of the word embeddings
    centroid = np.mean([model[word] for word in common_words], axis=0)

    # Find the closest word to the centroid
    closest_word = model.similar_by_vector(centroid, topn=1)[0][0]

    # Print the closest word
    print('sentiment : ', sentiment)
    
    print('\n')

/opt/conda/lib/python3.7/site-packages/nltk/twitter/__init__.py:20: UserWarning: The twython library has not been installed. Some functionality from the twitter package will not be available.
  warnings.warn("The twython library has not been installed. "


['evening', 'night', 'weekend', 'nights', 'morning']
sentiment :  0.5106


['room', 'rooms', 'bathroom', 'lounge']
sentiment :  0.9099


['bed', 'couch', 'sleep', 'beds']
sentiment :  0.7351


['pillows', 'mats', 'carpets']
sentiment :  -0.7003


['breakfast', 'dinner', 'lunch']
sentiment :  0.7351


['hotel', 'restaurant', 'hotels']
sentiment :  0.5859


['thing', 'things', 'guess']
sentiment :  0.7579


['city', 'town', 'downtown']
sentiment :  0.8934


['shops', 'restaurants', 'shop']
sentiment :  0.8519


['husband', 'friend', 'spouse']
sentiment :  0.8519




In [13]:
import pandas as pd

df = pd.read_csv("/kaggle/input/trip-advisor-hotel-reviews/tripadvisor_hotel_reviews.csv")
df = df.Review
df = df[:100]

noun_adj_df = pd.DataFrame(columns=['noun','adjective'])


nlp = spacy.load('en_core_web_sm')

for review in df:
    doc = nlp(review)

    for chunk in doc.noun_chunks:
        adj = []
        noun = ""
        for tok in chunk:
            if tok.pos_ == "NOUN":
                noun = tok.text
            if tok.pos_ == "ADJ":
                adj.append(tok.text)
        if noun and adj:
            if noun in model:
                new_row = pd.Series([noun,adj], index=noun_adj_df.columns)
                noun_adj_df = noun_adj_df.append(new_row, ignore_index=True)

                
topics = noun_adj_df
topics

,noun,adjective
0,parking,"[nice, expensive]"
1,deal,[good]
2,reviews,[previous]
3,room,"[easy, little, disappointed, non, -, existent]"
4,size,[nice]
...,...,...
693,stay,[horrible]
694,water,[hot]
695,shower,[cold]
696,problem,"[unhelpful, sure]"


In [14]:
nouns = topics['noun']
nouns

0      parking
1         deal
2      reviews
3         room
4         size
        ...   
693       stay
694      water
695     shower
696    problem
697       area
Name: noun, Length: 698, dtype: object

In [15]:
count =0
for item in nouns:
    if item == 'room':
        count += 1

print(count)

37


In [16]:
# Create an empty list to store the groups
groups = []
checked_words = []

for i in range (len(nouns)):
    temp=[]
    if i not in checked_words:
        temp.append(nouns[i])
        checked_words.append(i)

        for j in range (len(nouns)):
            if i==j:
                continue
            if j  in checked_words:
                continue
            if nouns[i] == nouns[j]:
                temp.append(nouns[j])
                checked_words.append(j)
                continue
            
            # Find the similarity score between the two words
            similarity = model.similarity(nouns[i] , nouns[j])

            # If the similarity score is greater than 0.8, add the second word to the group
            if similarity > 0.5:
                temp.append(nouns[j])
                checked_words.append(j)
    
        groups.append(temp)
    
groups

[['parking', 'parking', 'park', 'plaza'],
 ['deal', 'deals'],
 ['reviews', 'reviews', 'reviews'],
 ['room',
  'room',
  'room',
  'room',
  'rooms',
  'room',
  'room',
  'room',
  'room',
  'bathroom',
  'room',
  'rooms',
  'room',
  'rooms',
  'room',
  'room',
  'room',
  'lounge',
  'rooms',
  'room',
  'rooms',
  'room',
  'rooms',
  'room',
  'rooms',
  'room',
  'room',
  'room',
  'room',
  'bathroom',
  'bathroom',
  'rooms',
  'room',
  'rooms',
  'bathroom',
  'rooms',
  'rooms',
  'room',
  'rooms',
  'room',
  'room',
  'room',
  'rooms',
  'rooms',
  'room',
  'room',
  'room',
  'room',
  'bathroom',
  'room',
  'room',
  'room',
  'room',
  'room',
  'bathroom',
  'kitchenette',
  'room',
  'room',
  'room'],
 ['size', 'size', 'size'],
 ['pillows',
  'pillows',
  'pillows',
  'mats',
  'carpets',
  'carpets',
  'pillows',
  'towels',
  'blankets',
  'pillows'],
 ['neighbors', 'neighbors'],
 ['touch',
  'touch',
  'touch',
  'touches',
  'touch',
  'touch',
  'touches',

In [17]:
# Create an empty list to store the groups
groups = {}
checked_words = []

for i in range(len(nouns)):
    if i not in checked_words:
        temp = [nouns[i]]
        checked_words.append(i)

        for j in range(i + 1, len(nouns)):
            if j in checked_words:
                continue
            if nouns[i] == nouns[j]:
                temp.append(nouns[j])
                checked_words.append(j)
                continue
            
            # Find the similarity score between the two words
            similarity = model.similarity(nouns[i], nouns[j])

            # If the similarity score is greater than 0.8, add the second word to the group
            if similarity > 0.5:
                temp.append(nouns[j])
                checked_words.append(j)

        groups[i] = temp

groups = list(groups.values())
g =groups

In [18]:
g == groups

True

In [19]:
groups[3].count('room')

37

In [20]:
len(nouns)

698

In [21]:
total=0
for sublist in groups:
    total += len(sublist)

print(total)

698


In [22]:
len(checked_words)

698